In [1]:
import argparse
import pandas as pd
from openai import OpenAI

In [2]:
client = OpenAI(
    api_key=""
    )

def pipeline_gpt(msg, client, aspects):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", 
            "content": "You are an experienced annotator, who is performing multi-aspect annotation task based on the answer to a question asked during a mental health survey. You will need to annotate up to 3 most important aspects mentioned among the answer.\nThe questions in the mental health survey are as follow:\nCan you share your thoughts on the mental health or wellness services at your college? Identify aspects that are working well and areas that may need more attention.\n\nFor example, if the answer is 'I like the counseling services, but I wish they were more accessible,' the aspects are 'counseling services' and 'accessibility.', you only need to answer aspects split by comma. In some cases, the answer doesn't contain specific aspects, so you could simply use 'general' as the only aspect.\nAfter identifying the aspects, you should first choose the following pre-defined aspects that have similar meaning with any of aspects that you have identified:\n[" + aspects + "]\nOnly if you cannot find the similar meaning for some aspects, you can assign new one for this answer."
            },

            {
                "role": "user",
                "content": msg
            }
        ],
        temperature=0.1,
        top_p=0.5,
    )

    return response.choices[0].message.content

In [ ]:
df = pd.read_csv('smile-college-dataset.csv')

results = []
aspects = ''
for idx, row in df.iterrows():
    msg = row['comment']
    response = pipeline_gpt(msg, client, aspects)
    aspects += response + ', '
    results.append(response)
df['Aspects'] = results
df.to_csv('output.csv', index=False)